# AIPW and Double Machine Learning: From Theory to Code

**Shunyu Hao**, PLSC 40601, University of Chicago, Spring 2026

This tutorial covers the main ideas behind the Augmented Inverse Probability Weighting (AIPW) estimator and the Double Machine Learning (DML) framework. We start from the potential outcomes setup. We then build up to the AIPW estimator step by step. After that, we explain why Neyman orthogonality and cross-fitting matter. Finally, we implement DML from scratch and run a Monte Carlo simulation.

The goal is to help readers understand both the theory and the code. After reading this tutorial, you should know the strengths and limits of DML, and you should be able to implement it yourself.

# 1 Background: The Average Treatment Effect

## 1.1 Setup

We use the potential outcomes framework (Rubin 1974). We observe $n$ units. Each unit $i$ has:

- $D_i \in \{0, 1\}$: treatment status (1 = treated, 0 = control)
- $Y_i(0), Y_i(1)$: potential outcomes under control and treatment
- $X_i \in \mathbb{R}^p$: pre-treatment covariates

We only see one potential outcome for each unit. The observed outcome is:

$$Y_i = D_i \cdot Y_i(1) + (1 - D_i) \cdot Y_i(0)$$

The **Average Treatment Effect** (ATE) is:

$$\tau = \mathbb{E}[Y(1) - Y(0)]$$

## 1.2 Why Simple Comparison Fails

In observational studies, treated and control groups may differ before treatment. A simple comparison of group means gives:

$$\mathbb{E}[Y \mid D=1] - \mathbb{E}[Y \mid D=0] = \underbrace{\mathbb{E}[Y(1)-Y(0) \mid D=1]}_{\text{ATT}} + \underbrace{\mathbb{E}[Y(0)\mid D=1] - \mathbb{E}[Y(0)\mid D=0]}_{\text{Selection Bias}}$$

The second term is selection bias. We need extra assumptions to remove it.

## 1.3 Identification Assumptions

**Assumption 1 (Unconfoundedness):** $(Y(0), Y(1)) \perp\!\!\!\perp D \mid X$

This says that treatment is as good as random after we condition on $X$.

**Assumption 2 (Overlap):** There exists $\eta > 0$ such that $\eta \le e(x) \le 1 - \eta$ for all $x$, where $e(x) = P(D=1 \mid X=x)$ is the propensity score.

This says that every unit has a chance to be treated or untreated.

## 1.4 Three Identification Formulas

Define $\mu_d(x) = \mathbb{E}[Y \mid D=d, X=x]$. Under the assumptions above, we get three ways to identify $\tau$:

**Outcome Regression (OR):**
$$\tau = \mathbb{E}_X[\mu_1(X) - \mu_0(X)]$$

**Inverse Probability Weighting (IPW):**
$$\tau = \mathbb{E}\left[\frac{DY}{e(X)} - \frac{(1-D)Y}{1-e(X)}\right]$$

**Augmented IPW (AIPW):**
$$\tau = \mathbb{E}\left[\mu_1(X) - \mu_0(X) + \frac{D(Y - \mu_1(X))}{e(X)} - \frac{(1-D)(Y - \mu_0(X))}{1-e(X)}\right]$$

OR uses only the outcome model. IPW uses only the propensity score. AIPW uses both.

## 1.5 Code: Generate Data and Show Selection Bias

We now write a simple data generating process (DGP). Treatment depends on $X$, so a naive comparison is biased.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import KFold
from sklearn.linear_model import LinearRegression, LogisticRegression, Lasso
from sklearn.ensemble import RandomForestRegressor, RandomForestClassifier
from sklearn.ensemble import GradientBoostingRegressor, GradientBoostingClassifier
from sklearn.neural_network import MLPRegressor, MLPClassifier
from sklearn.base import clone
import warnings
warnings.filterwarnings("ignore")

np.random.seed(42)

In [ ]:
# generate data with confounding
nobs = 500
p = 5
truetau = 2.0

x = np.random.normal(0, 1, (nobs, p))

# outcome model: mu0 depends on x
beta = np.array([1.0, 0.8, 0.6, 0.0, 0.0])
mu0true = x @ beta
mu1true = mu0true + truetau

# propensity score depends on x
gamma = np.array([0.5, 0.4, 0.3, 0.0, 0.0])
logit = x @ gamma
etrue = 1.0 / (1.0 + np.exp(-logit))
etrue = np.clip(etrue, 0.05, 0.95)

# generate treatment and outcome
d = np.random.binomial(1, etrue)
y = d * mu1true + (1 - d) * mu0true + np.random.normal(0, 1, nobs)

print(f"True ATE: {truetau}")
print(f"Naive difference in means: {y[d==1].mean() - y[d==0].mean():.3f}")
print("The naive estimate is biased because treated units have higher X values.")

# 2 Three Estimation Strategies

## 2.1 Outcome Regression (OR)

The OR estimator fits models $\hat{\mu}_1$ and $\hat{\mu}_0$, then averages their difference:

$$\hat{\tau}_{OR} = \frac{1}{n} \sum_{i=1}^{n} [\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i)]$$

It is consistent only if the outcome model is correct.

## 2.2 IPW Estimator

The IPW estimator reweights outcomes by the propensity score:

$$\hat{\tau}_{IPW} = \frac{1}{n} \sum_{i=1}^{n} \left[\frac{D_i Y_i}{\hat{e}(X_i)} - \frac{(1-D_i) Y_i}{1-\hat{e}(X_i)}\right]$$

It is consistent only if the propensity score model is correct. It also has high variance when $e(x)$ is close to 0 or 1.

## 2.3 AIPW Estimator

The AIPW estimator combines both models:

$$\hat{\tau}_{AIPW} = \frac{1}{n} \sum_{i=1}^{n} \left[\hat{\mu}_1(X_i) - \hat{\mu}_0(X_i) + \frac{D_i(Y_i - \hat{\mu}_1(X_i))}{\hat{e}(X_i)} - \frac{(1-D_i)(Y_i - \hat{\mu}_0(X_i))}{1-\hat{e}(X_i)}\right]$$

AIPW is **doubly robust**: it is consistent if either the outcome model or the propensity score model is correct. You do not need both to be right.

**Why is this true?** If the outcome model is correct, the residual $Y - \hat{\mu}_d(X)$ has mean zero. The IPW correction term vanishes. If the propensity score is correct, the IPW correction exactly fixes the bias from a wrong outcome model. This \"double insurance\" makes AIPW safer than OR or IPW alone.

## 2.4 Code: Compare OR, IPW, and AIPW

In [ ]:
# fit outcome models
idx0 = np.where(d == 0)[0]
idx1 = np.where(d == 1)[0]

reg0 = LinearRegression().fit(x[idx0], y[idx0])
reg1 = LinearRegression().fit(x[idx1], y[idx1])
mu0hat = reg0.predict(x)
mu1hat = reg1.predict(x)

# fit propensity score
psmodel = LogisticRegression(max_iter=1000).fit(x, d)
ehat = psmodel.predict_proba(x)[:, 1]
ehat = np.clip(ehat, 0.01, 0.99)

# OR estimator
tauor = np.mean(mu1hat - mu0hat)

# IPW estimator
tauipw = np.mean(d * y / ehat - (1 - d) * y / (1 - ehat))

# AIPW estimator
scores = (mu1hat - mu0hat
          + d * (y - mu1hat) / ehat
          - (1 - d) * (y - mu0hat) / (1 - ehat))
tauaipw = np.mean(scores)

print(f"True ATE:       {truetau:.3f}")
print(f"OR estimate:    {tauor:.3f}")
print(f"IPW estimate:   {tauipw:.3f}")
print(f"AIPW estimate:  {tauaipw:.3f}")

# 3 The Efficient Influence Function

## 3.1 Why It Matters

Our problem is **semiparametric**. We care about a single number $\tau$, but the nuisance functions $\mu_0, \mu_1, e$ are infinite-dimensional. The efficient influence function (EIF) tells us the best we can do.

## 3.2 The EIF for the ATE

The EIF for the ATE is (Hahn 1998):

$$\varphi(Y, D, X) = \mu_1(X) - \mu_0(X) - \tau + \frac{D(Y - \mu_1(X))}{e(X)} - \frac{(1-D)(Y - \mu_0(X))}{1 - e(X)}$$

This is exactly the AIPW score. The AIPW estimator solves $\frac{1}{n}\sum_i \varphi_i = 0$ for $\tau$. This is not a coincidence. The EIF gives the estimator with the smallest possible variance.

## 3.3 The Efficiency Bound

Any well-behaved estimator of $\tau$ must have asymptotic variance at least:

$$V_{\text{eff}} = \mathbb{E}\left[\frac{\sigma_1^2(X)}{e(X)} + \frac{\sigma_0^2(X)}{1-e(X)} + (\tau(X) - \tau)^2\right]$$

where $\sigma_d^2(x) = \text{Var}(Y(d) \mid X=x)$ and $\tau(x) = \mu_1(x) - \mu_0(x)$.

When both $\hat{\mu}_d$ and $\hat{e}$ are consistent, AIPW reaches this bound. It is efficient.

# 4 Neyman Orthogonality

## 4.1 The Problem with Plug-in Estimators

Suppose we define $\tau$ by a moment condition: $\mathbb{E}[\psi(W; \tau, \eta)] = 0$, where $\eta = (\mu_0, \mu_1, e)$ are nuisance parameters. When we plug in $\hat{\eta}$, a Taylor expansion gives:

$$\mathbb{E}[\psi(W; \tau, \hat{\eta})] \approx \underbrace{\frac{\partial}{\partial \eta}\mathbb{E}[\psi]}_{\text{first-order term}} \cdot (\hat{\eta} - \eta_0) + O(\|\hat{\eta} - \eta_0\|^2)$$

If the first-order term is not zero, errors in $\hat{\eta}$ pass directly into $\hat{\tau}$. This requires $\hat{\eta}$ to converge at rate $n^{-1/2}$. Machine learning methods usually cannot do this.

## 4.2 Definition

A moment function $\psi(W; \tau, \eta)$ satisfies **Neyman orthogonality** if:

$$\frac{\partial}{\partial t}\mathbb{E}[\psi(W; \tau_0, \eta_0 + t(\eta - \eta_0))]\bigg|_{t=0} = 0$$

This means the moment is \"flat\" with respect to $\eta$ at the true value. First-order errors in $\hat{\eta}$ do not affect $\hat{\tau}$. Only second-order errors matter.

## 4.3 AIPW Is Neyman Orthogonal

The AIPW moment satisfies Neyman orthogonality. When we change $\mu_1$ by a small amount $\delta$, the direct effect ($+\delta$) cancels with the indirect effect ($-\delta$) through the IPW correction. The derivative with respect to $e$ is also zero because $\mathbb{E}[Y - \mu_1(X) \mid D=1, X] = 0$.

## 4.4 Counterexample: IPW Is Not Orthogonal

The IPW moment does **not** satisfy Neyman orthogonality. Its derivative with respect to $e$ is not zero. So first-order errors in $\hat{e}$ directly cause bias in $\hat{\tau}_{IPW}$. This is why IPW works poorly with ML-estimated propensity scores.

## 4.5 Code: Show That IPW Bias Grows with Propensity Score Error

We add noise to the true propensity score and compare IPW and AIPW. AIPW is much more stable because it is Neyman orthogonal.

In [ ]:
# show robustness of AIPW vs IPW to propensity score error
noiselevels = [0.0, 0.1, 0.2, 0.5, 1.0, 2.0]
ipwresults = []
aipwresults = []

for nl in noiselevels:
    enoisy = etrue + nl * np.random.normal(0, 1, nobs)
    enoisy = np.clip(enoisy, 0.02, 0.98)

    # IPW with noisy propensity score
    tipw = np.mean(d * y / enoisy - (1 - d) * y / (1 - enoisy))
    ipwresults.append(tipw)

    # AIPW with noisy propensity score but correct outcome model
    sc = (mu1true - mu0true
          + d * (y - mu1true) / enoisy
          - (1 - d) * (y - mu0true) / (1 - enoisy))
    aipwresults.append(np.mean(sc))

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(noiselevels, ipwresults, "o-", label="IPW", color="tomato")
ax.plot(noiselevels, aipwresults, "s-", label="AIPW", color="steelblue")
ax.axhline(truetau, color="black", linestyle="--", label=f"True ATE = {truetau}")
ax.set_xlabel("Noise level added to propensity score")
ax.set_ylabel("Estimated ATE")
ax.set_title("AIPW is robust to propensity score error (Neyman orthogonality)")
ax.legend()
plt.tight_layout()
plt.savefig("fig1_orthogonality.png", dpi=150)
plt.show()
print("AIPW stays close to the true ATE even when the propensity score is noisy.")
print("IPW becomes very biased. This is because AIPW is Neyman orthogonal.")

# 5 Double Machine Learning (DML)

This is the main section of the tutorial. We build DML step by step.

The DML framework (Chernozhukov et al. 2018) has three parts:

1. Use a **Neyman orthogonal moment** (the AIPW score)
2. Use **machine learning** to estimate the nuisance functions $\hat{\mu}_0, \hat{\mu}_1, \hat{e}$
3. Use **cross-fitting** to avoid overfitting bias

We now explain each part and write the code.

## 5.1 Why We Need Cross-Fitting

If we train $\hat{\mu}$ and $\hat{e}$ on the full sample and then compute AIPW scores on the same sample, there is a problem. The ML models may overfit. The scores are computed using models that already saw observation $i$. This creates a dependence that breaks the theory.

Cross-fitting solves this. We split the data into $K$ folds. For each fold $k$, we train the ML models on all data except fold $k$. Then we compute scores only on fold $k$. This way, the model never sees the data it is scoring.

## 5.2 The DML Algorithm

Here is the full algorithm:

**Step 1.** Split the sample into $K$ folds $I_1, \ldots, I_K$.

**Step 2.** For each fold $k$: train $\hat{\mu}_0^{(-k)}$ on control units in other folds, train $\hat{\mu}_1^{(-k)}$ on treated units in other folds, train $\hat{e}^{(-k)}$ on all units in other folds.

**Step 3.** For each unit $i$ in fold $k$, compute the AIPW score:
$$\hat{\varphi}_i = \hat{\mu}_1^{(-k)}(X_i) - \hat{\mu}_0^{(-k)}(X_i) + \frac{D_i(Y_i - \hat{\mu}_1^{(-k)}(X_i))}{\hat{e}^{(-k)}(X_i)} - \frac{(1-D_i)(Y_i - \hat{\mu}_0^{(-k)}(X_i))}{1 - \hat{e}^{(-k)}(X_i)}$$

**Step 4.** The DML estimate is the average of all scores:
$$\hat{\tau}_{DML} = \frac{1}{n} \sum_{i=1}^{n} \hat{\varphi}_i$$

**Step 5.** The standard error is:
$$\hat{se} = \sqrt{\frac{1}{n^2} \sum_{i=1}^{n} (\hat{\varphi}_i - \hat{\tau}_{DML})^2}$$

## 5.3 The Product Rate Condition

DML gives valid inference under a key condition:

$$\|\hat{\mu}_d - \mu_d\|_{L^2} \cdot \|\hat{e} - e\|_{L^2} = o_p(n^{-1/2})$$

This is the **product rate condition**. The product of the two estimation errors must go to zero faster than $n^{-1/2}$.

If each nuisance estimator converges at rate $n^{-1/4}$, the product is $n^{-1/4} \times n^{-1/4} = n^{-1/2}$. So each ML model only needs $n^{-1/4}$ rate. This is much easier than $n^{-1/2}$. Many ML methods can achieve $n^{-1/4}$.

Without Neyman orthogonality, we would need $n^{-1/2}$ rate for each nuisance. Most ML methods cannot do that. This is why orthogonality matters so much.

## 5.4 Code: DML from Scratch, Step by Step

We now implement the full DML procedure. We go through each step slowly with comments.

In [ ]:
# =========================================================
# STEP 0: generate fresh data for the DML example
# =========================================================

np.random.seed(123)
nobs = 1000
p = 5
truetau = 2.0

x = np.random.normal(0, 1, (nobs, p))
beta = np.array([1.0, 0.8, 0.6, 0.4, 0.2])
mu0true = x @ beta
mu1true = mu0true + truetau

gamma = np.array([0.5, 0.4, 0.3, 0.2, 0.1])
logit = x @ gamma
etrue = 1.0 / (1.0 + np.exp(-logit))
etrue = np.clip(etrue, 0.05, 0.95)

d = np.random.binomial(1, etrue)
y = d * mu1true + (1 - d) * mu0true + np.random.normal(0, 1, nobs)

print(f"Data generated: n = {nobs}, p = {p}, true ATE = {truetau}")
print(f"Number treated: {d.sum()}, number control: {(1-d).sum()}")

In [ ]:
# =========================================================
# STEP 1: set up K-fold cross-fitting
# =========================================================

nfolds = 5
kf = KFold(n_splits=nfolds, shuffle=True, random_state=42)

# we will store predictions for every observation
mu0hat = np.zeros(nobs)
mu1hat = np.zeros(nobs)
ehat = np.zeros(nobs)

print(f"We split the data into {nfolds} folds.")
print(f"Each fold has about {nobs // nfolds} observations.")
print()

foldnum = 0
for trainidx, testidx in kf.split(x):
    foldnum += 1
    print(f"Fold {foldnum}: train on {len(trainidx)} obs, predict on {len(testidx)} obs")

print()
print("For each fold, we train on the other folds and predict on this fold.")
print("This is cross-fitting. The model never scores data it trained on.")

In [ ]:
# =========================================================
# STEP 2: for each fold, train models and make predictions
# =========================================================

# we use Lasso for the outcome models and logistic regression
# for the propensity score

nfolds = 5
kf = KFold(n_splits=nfolds, shuffle=True, random_state=42)

mu0hat = np.zeros(nobs)
mu1hat = np.zeros(nobs)
ehat = np.zeros(nobs)

foldnum = 0
for trainidx, testidx in kf.split(x):
    foldnum += 1

    # separate treated and control in the training set
    traincontrol = trainidx[d[trainidx] == 0]
    traintreated = trainidx[d[trainidx] == 1]

    # train outcome model for control group
    model0 = Lasso(alpha=0.05, max_iter=2000)
    model0.fit(x[traincontrol], y[traincontrol])

    # train outcome model for treated group
    model1 = Lasso(alpha=0.05, max_iter=2000)
    model1.fit(x[traintreated], y[traintreated])

    # train propensity score model
    psmod = LogisticRegression(penalty="l1", C=1.0, solver="saga", max_iter=2000)
    psmod.fit(x[trainidx], d[trainidx])

    # predict on the held-out fold
    mu0hat[testidx] = model0.predict(x[testidx])
    mu1hat[testidx] = model1.predict(x[testidx])
    ehat[testidx] = psmod.predict_proba(x[testidx])[:, 1]

    ntrain = len(traincontrol) + len(traintreated)
    print(f"Fold {foldnum}: trained on {len(traincontrol)} ctrl + {len(traintreated)} treat, predicted {len(testidx)}")

# clip propensity scores to avoid extreme weights
ehat = np.clip(ehat, 0.01, 0.99)

print()
print("All folds done. Every observation now has out-of-sample predictions.")

In [ ]:
# =========================================================
# STEP 3: compute the AIPW scores for each observation
# =========================================================

# the AIPW score for observation i is:
# phi_i = mu1hat(xi) - mu0hat(xi)
#        + di * (yi - mu1hat(xi)) / ehat(xi)
#        - (1 - di) * (yi - mu0hat(xi)) / (1 - ehat(xi))

# let us compute this for one observation first, to be clear
i = 0
orpart = mu1hat[i] - mu0hat[i]
if d[i] == 1:
    treatedpart = (y[i] - mu1hat[i]) / ehat[i]
    controlpart = 0.0
else:
    treatedpart = 0.0
    controlpart = (y[i] - mu0hat[i]) / (1 - ehat[i])

phi0 = orpart + treatedpart - controlpart
print(f"Score for observation 0: {phi0:.4f}")
print(f"  OR part:      {orpart:.4f}")
print(f"  Treated part: {treatedpart:.4f}")
print(f"  Control part: {controlpart:.4f}")
print()

# now compute all scores at once using vectorized code
scores = (mu1hat - mu0hat
          + d * (y - mu1hat) / ehat
          - (1 - d) * (y - mu0hat) / (1 - ehat))

print(f"Computed AIPW scores for all {nobs} observations.")
print(f"Score mean: {scores.mean():.4f}")
print(f"Score std:  {scores.std():.4f}")

In [ ]:
# =========================================================
# STEP 4: compute the DML estimate and standard error
# =========================================================

taudml = np.mean(scores)

# standard error using the EIF variance formula
variance = np.mean((scores - taudml) ** 2)
se = np.sqrt(variance / nobs)

# 95% confidence interval
cilo = taudml - 1.96 * se
cihi = taudml + 1.96 * se

print("=" * 50)
print("DML RESULTS")
print("=" * 50)
print(f"True ATE:              {truetau:.4f}")
print(f"DML estimate:          {taudml:.4f}")
print(f"Standard error:        {se:.4f}")
print(f"95% CI:                [{cilo:.4f}, {cihi:.4f}]")
print(f"CI covers true value:  {cilo <= truetau <= cihi}")
print("=" * 50)

## 5.5 Code: Wrap DML into a Reusable Function

We now put all the steps above into one function. This makes it easy to reuse with different learners.

In [ ]:
def dml(x, d, y, model0, model1, psmodel, nfolds=5, seed=42):
    # Run DML with cross-fitting.
    # x: covariates (n, p), d: treatment (n,), y: outcome (n,)
    # model0, model1: sklearn regressors for outcome
    # psmodel: sklearn classifier for propensity score
    # Returns: tauhat, se, scores

    n = len(y)
    kf = KFold(n_splits=nfolds, shuffle=True, random_state=seed)

    mu0h = np.zeros(n)
    mu1h = np.zeros(n)
    eh = np.zeros(n)

    for trainidx, testidx in kf.split(x):
        trainctrl = trainidx[d[trainidx] == 0]
        traintreat = trainidx[d[trainidx] == 1]

        m0 = clone(model0)
        m1 = clone(model1)
        ps = clone(psmodel)

        m0.fit(x[trainctrl], y[trainctrl])
        m1.fit(x[traintreat], y[traintreat])
        ps.fit(x[trainidx], d[trainidx])

        mu0h[testidx] = m0.predict(x[testidx])
        mu1h[testidx] = m1.predict(x[testidx])
        eh[testidx] = ps.predict_proba(x[testidx])[:, 1]

    eh = np.clip(eh, 0.01, 0.99)

    scores = (mu1h - mu0h
              + d * (y - mu1h) / eh
              - (1 - d) * (y - mu0h) / (1 - eh))

    tauhat = np.mean(scores)
    var = np.mean((scores - tauhat) ** 2)
    se = np.sqrt(var / n)

    return tauhat, se, scores, mu0h, mu1h, eh


# test the function
m0 = Lasso(alpha=0.05, max_iter=2000)
m1 = Lasso(alpha=0.05, max_iter=2000)
ps = LogisticRegression(penalty="l1", C=1.0, solver="saga", max_iter=2000)

tauhat, se, sc, _, _, _ = dml(x, d, y, m0, m1, ps)
print(f"DML function test: tauhat = {tauhat:.4f}, SE = {se:.4f}")
print(f"True ATE: {truetau}")

# 6 Monte Carlo Simulation: How Nuisance Learner Choice Affects DML

The theory says DML works when the product rate condition holds. But in practice, does the choice of ML method matter? We now run a simulation to find out.

We test four ML methods under two DGPs.

## 6.1 Two Data Generating Processes

**DGP 1: Sparse and High-Dimensional (p = 100).** The outcome and propensity score depend on only 5 out of 100 covariates. The relationships are linear. This is a good match for Lasso.

**DGP 2: Smooth and Nonlinear (p = 10).** The outcome has nonlinear terms like $\sin(\pi X_1 X_2)$ and $(X_3 - 0.5)^2$. This is a good match for tree methods and neural networks.

## 6.2 Four Nuisance Learners

Lasso, Random Forest, Gradient Boosting, and Neural Network.

In [ ]:
def dgpsparse(n=1000, p=100, seed=None):
    rng = np.random.default_rng(seed)
    x = rng.standard_normal((n, p))
    beta = np.zeros(p)
    beta[:5] = [1.0, 0.8, 0.6, 0.4, 0.2]
    gamma = np.zeros(p)
    gamma[:5] = [0.5, 0.4, 0.3, 0.2, 0.1]
    tau = 2.0
    mu0 = x @ beta
    mu1 = mu0 + tau
    logit = x @ gamma
    e = 1.0 / (1.0 + np.exp(-logit))
    e = np.clip(e, 0.05, 0.95)
    d = rng.binomial(1, e)
    y = d * mu1 + (1 - d) * mu0 + rng.standard_normal(n)
    return x, d, y, tau, mu0, mu1, e


def dgpsmooth(n=1000, p=10, seed=None):
    rng = np.random.default_rng(seed)
    x = rng.uniform(-1, 1, (n, p))
    tau = 2.0
    mu0 = (np.sin(np.pi * x[:, 0] * x[:, 1])
           + 2 * (x[:, 2] - 0.5) ** 2
           + x[:, 3] + 0.5 * x[:, 4]
           + 0.3 * np.cos(x[:, 5] * x[:, 6])
           + 0.2 * x[:, 7] + 0.1 * x[:, 8] * x[:, 9])
    mu1 = mu0 + tau
    logite = (np.sin(np.pi * x[:, 0]) + x[:, 1] ** 2
              + x[:, 2] + 0.5 * x[:, 3] * x[:, 4] + 0.3 * x[:, 5])
    e = 1.0 / (1.0 + np.exp(-logite))
    e = np.clip(e, 0.05, 0.95)
    d = rng.binomial(1, e)
    y = d * mu1 + (1 - d) * mu0 + rng.standard_normal(n)
    return x, d, y, tau, mu0, mu1, e

print("DGP functions defined.")

In [ ]:
try:
    from lightgbm import LGBMRegressor, LGBMClassifier
    haslgbm = True
except ImportError:
    haslgbm = False
    print("lightgbm not found, using sklearn GradientBoosting instead")


def getlearners(name):
    if name == "Lasso":
        m0 = Lasso(alpha=0.05, max_iter=2000)
        m1 = Lasso(alpha=0.05, max_iter=2000)
        ps = LogisticRegression(penalty="l1", C=1.0, solver="saga", max_iter=2000)
    elif name == "Random Forest":
        kw = dict(n_estimators=100, max_depth=8, min_samples_leaf=5, random_state=42, n_jobs=-1)
        m0 = RandomForestRegressor(**kw)
        m1 = RandomForestRegressor(**kw)
        ps = RandomForestClassifier(**kw)
    elif name == "Gradient Boosting":
        if haslgbm:
            kw = dict(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42, verbosity=-1, n_jobs=-1)
            m0 = LGBMRegressor(**kw)
            m1 = LGBMRegressor(**kw)
            ps = LGBMClassifier(**kw)
        else:
            kw = dict(n_estimators=100, max_depth=4, learning_rate=0.1, random_state=42)
            m0 = GradientBoostingRegressor(**kw)
            m1 = GradientBoostingRegressor(**kw)
            ps = GradientBoostingClassifier(**kw)
    elif name == "Neural Network":
        kw = dict(hidden_layer_sizes=(32,), max_iter=200, early_stopping=True, random_state=42)
        m0 = MLPRegressor(**kw)
        m1 = MLPRegressor(**kw)
        ps = MLPClassifier(**kw)
    else:
        raise ValueError(name)
    return m0, m1, ps

print("Learner factory defined.")

In [ ]:
def onesim(dgpfunc, learnername, seed):
    x, d, y, tau, mu0, mu1, e = dgpfunc(seed=seed)
    m0, m1, ps = getlearners(learnername)
    tauhat, se, sc, mu0h, mu1h, eh = dml(x, d, y, m0, m1, ps, seed=seed)

    mutrue = d * mu1 + (1 - d) * mu0
    muhat = d * mu1h + (1 - d) * mu0h
    rmsemu = np.sqrt(np.mean((muhat - mutrue) ** 2))
    rmsee = np.sqrt(np.mean((eh - e) ** 2))

    cilo = tauhat - 1.96 * se
    cihi = tauhat + 1.96 * se

    return {
        "tauhat": tauhat, "se": se,
        "bias": tauhat - tau,
        "covers": int(cilo <= tau <= cihi),
        "rmsemu": rmsemu, "rmsee": rmsee,
        "product": rmsemu * rmsee,
    }

print("Single simulation function defined.")

In [ ]:
import time

nsims = 50
learnernames = ["Lasso", "Random Forest", "Gradient Boosting", "Neural Network"]
dgplist = [("Sparse (p=100)", dgpsparse), ("Smooth (p=10)", dgpsmooth)]

allrows = []
for dgpname, dgpfunc in dgplist:
    for ln in learnernames:
        t0 = time.time()
        print(f"Running: {dgpname} x {ln} ({nsims} reps)...", end=" ", flush=True)
        for s in range(nsims):
            res = onesim(dgpfunc, ln, seed=2026 + s)
            res["dgp"] = dgpname
            res["learner"] = ln
            allrows.append(res)
        elapsed = time.time() - t0
        print(f"done in {elapsed:.0f}s")

results = pd.DataFrame(allrows)
print(f"\nTotal simulations: {len(results)}")

In [ ]:
summary = results.groupby(["dgp", "learner"]).agg(
    bias=("bias", "mean"),
    sd=("tauhat", "std"),
    rmsetau=("bias", lambda x: np.sqrt(np.mean(x ** 2))),
    coverage=("covers", "mean"),
    meanse=("se", "mean"),
    rmsemu=("rmsemu", "mean"),
    rmsee=("rmsee", "mean"),
    product=("product", "mean"),
).reset_index()

pd.set_option("display.float_format", lambda x: f"{x:.4f}")
print("=" * 90)
print("SIMULATION SUMMARY")
print("=" * 90)
print(summary.to_string(index=False))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(15, 9))
metrics = [("bias", "Bias"), ("coverage", "95% CI Coverage"),
           ("rmsetau", "RMSE of ATE estimate"), ("rmsemu", "RMSE of outcome model"),
           ("rmsee", "RMSE of propensity score"), ("product", "Product rate")]

dgps = summary["dgp"].unique()
ls = summary["learner"].unique()
xpos = np.arange(len(ls))
w = 0.35

for idx, (col, title) in enumerate(metrics):
    ax = axes[idx // 3, idx % 3]
    for j, dgp in enumerate(dgps):
        sub = summary[summary["dgp"] == dgp]
        vals = [sub.loc[sub["learner"] == l, col].values[0] for l in ls]
        ax.bar(xpos + (j - 0.5) * w, vals, w, label=dgp, alpha=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_xticks(xpos)
    ax.set_xticklabels(ls, rotation=25, ha="right", fontsize=8)
    ax.legend(fontsize=7)
    ax.grid(axis="y", alpha=0.3)
    if col == "coverage":
        ax.axhline(0.95, color="red", linestyle="--", linewidth=1)
        ax.set_ylim(0.5, 1.05)
    if col == "bias":
        ax.axhline(0, color="gray", linestyle="--", linewidth=0.8)

fig.suptitle("DML Simulation Results", fontsize=14, fontweight="bold")
plt.tight_layout()
plt.savefig("fig2_simulation.png", dpi=150, bbox_inches="tight")
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))
for ax, dgp in zip(axes, dgps):
    for l in ls:
        sub = results[(results["dgp"] == dgp) & (results["learner"] == l)]
        ax.hist(sub["bias"], bins=20, alpha=0.4, label=l, density=True)
    ax.axvline(0, color="black", linestyle="--", linewidth=1)
    ax.set_title(f"Bias distribution: {dgp}", fontweight="bold")
    ax.set_xlabel("Bias (tauhat minus tau)")
    ax.set_ylabel("Density")
    ax.legend(fontsize=8)

plt.tight_layout()
plt.savefig("fig3_bias_dist.png", dpi=150, bbox_inches="tight")
plt.show()

# 7 Discussion and Takeaways

## What We Learned from the Simulation

**The product rate condition is the key.** When a learner fits the DGP well, both the outcome model and the propensity score are estimated well. The product of their errors is small. DML gives low bias and good coverage. When a learner does not fit the DGP, the product rate condition fails. Bias is large and confidence intervals are wrong.

**Learner choice matters a lot.** Under the sparse DGP, Lasso works best because the true model is linear and sparse. Under the smooth DGP, tree methods and neural networks work well because they can capture nonlinear patterns. There is no single best learner for all problems.

**Bias is the main problem, not variance.** When the product rate condition fails, the bias is much larger than the standard deviation. The standard error formula also fails. It underestimates the true uncertainty. So the confidence intervals have poor coverage.

**DML is not automatic.** The theory is elegant, but it only works when its conditions hold. In practice, you should: (1) choose a learner that matches your data structure, (2) check the cross-validation accuracy of your nuisance models, and (3) try multiple learners as a sensitivity check.

## Summary of the Three Pillars of DML

**Neyman orthogonality** makes the AIPW moment insensitive to first-order errors in the nuisance estimates. This relaxes the convergence rate requirement from $n^{-1/2}$ to $n^{-1/4}$.

**Cross-fitting** removes the need for the Donsker condition. It allows us to use complex ML methods.

**The product rate condition** requires the product of the outcome model error and the propensity score error to be $o_p(n^{-1/2})$. This is the condition under which DML gives valid confidence intervals.

## References

1. Chernozhukov, V. et al. (2018). Double/debiased machine learning for treatment and structural parameters. *The Econometrics Journal*, 21(1), C1-C68.
2. Hahn, J. (1998). On the role of the propensity score in efficient semiparametric estimation of average treatment effects. *Econometrica*, 66(2), 315-331.
3. Kennedy, E. H. (2022). Semiparametric doubly robust targeted double machine learning: A review. arXiv:2203.06469.
4. Robins, J. M., Rotnitzky, A. and Zhao, L. P. (1994). Estimation of regression coefficients when some regressors are not always observed. *JASA*, 89(427), 846-866.
5. Scharfstein, D. O., Rotnitzky, A. and Robins, J. M. (1999). Adjusting for nonignorable drop-out using semiparametric nonresponse models. *JASA*, 94(448), 1096-1120.